# 02 — Locked 2621 SPX VRP Signal and Final Per-Trade Sizing

This is the final cleaned notebook for the locked SPX VRP simple signal framework.

## Locked research candidate

`locked_2621_win_band_25bps_conditional`

## What this notebook does

1. Loads the production feature panel and naked ATM put outcome panel.
2. Validates the source data and builds a complete date × tenor research matrix.
3. Applies the locked Core / Secondary 2621 signal thresholds.
4. Selects at most one tenor per trade date using conditional win probability first, with a 25 bps win-band tie-breaker based on conditional average P&L/day.
5. Applies the locked per-trade max-risk sizing schedule.
6. Produces a cumulative realized P&L chart using a constant $1mm NAV sizing base.

## What this notebook intentionally excludes

- Search sweeps over alternative threshold templates.
- Fractional Kelly exploration.
- Portfolio overlap caps and hedge rules.
- Compounded NAV sizing.
- Exploratory validation tables that were used to arrive at the locked schedule.

Those items belong in research/audit notebooks. This notebook is the clean locked implementation path.


In [ ]:
# Cell 1 — Setup, imports, paths, and locked project metadata

from pathlib import Path
import json
import math
import warnings

import numpy as np
import pandas as pd

pd.set_option("display.max_columns", 200)
pd.set_option("display.width", 260)
pd.set_option("display.float_format", lambda x: f"{x:,.6f}")

warnings.filterwarnings("ignore", category=FutureWarning)

# ---------------------------------------------------------------------
# Project paths
# ---------------------------------------------------------------------
# This notebook is intended to run from either the project root or a
# notebooks/ subfolder. If neither location has a data/ directory, it falls
# back to the project path used during development.
# ---------------------------------------------------------------------

cwd = Path.cwd()

if (cwd / "data").exists():
    PROJECT_ROOT = cwd
elif (cwd.parent / "data").exists():
    PROJECT_ROOT = cwd.parent
else:
    PROJECT_ROOT = Path(r"C:\Users\patri\vrp_project")

DATA_DIR = PROJECT_ROOT / "data"
PROCESSED_DIR = DATA_DIR / "processed"
AUDIT_DIR = DATA_DIR / "audit"

PROCESSED_DIR.mkdir(parents=True, exist_ok=True)
AUDIT_DIR.mkdir(parents=True, exist_ok=True)

FEATURE_PANEL_PATH = PROCESSED_DIR / "production_feature_panel_v0_1.parquet"
OUTCOME_PATH = PROCESSED_DIR / "naked_atm_put_eod_panel_v0_1.parquet"

# ---------------------------------------------------------------------
# Tenor universe and grouping
# ---------------------------------------------------------------------

PRODUCTION_TENORS = [9, 12, 15, 18, 21, 24, 27, 30, 33]

GROUP_TENORS = {
    "front": [9, 12, 15],
    "middle": [18, 21, 24],
    "back": [27, 30, 33],
}

TENOR_GROUP_MAP = {
    tenor: group
    for group, tenors in GROUP_TENORS.items()
    for tenor in tenors
}

# ---------------------------------------------------------------------
# Locked model metadata
# ---------------------------------------------------------------------
# Current preferred research candidate:
#   locked_2621_win_band_25bps_conditional
#
# Important conventions:
#   - Thresholds are bucketed front / middle / back, not line-fitted.
#   - Core is checked before Secondary.
#   - Secondary is checked only when no Core tenor qualifies.
#   - Within the active layer, tenor priority is based on layer-specific
#     2621-conditional win probability.
#   - Tenors within 25 bps of the best conditional win probability are treated
#     as near-ties; conditional average P&L/day chooses among that near-tie set.
#   - Raw average P&L is not used for tenor priority.
# ---------------------------------------------------------------------

MODEL_LABEL = "locked_2621_win_band_25bps_conditional"
WIN_BAND = 0.0025

# Final locked per-trade max-risk sizing, as a fraction of constant NAV.
# This is separate from portfolio overlap, stress caps, and hedging.
LOCKED_PER_TRADE_RISK_FRACTION = {
    "Core_back": 0.0500,
    "Core_middle": 0.0485,
    "Core_front": 0.0175,
    "Secondary_back": 0.0450,
    "Secondary_middle": 0.0325,
    "Secondary_front": 0.0175,
}

CONSTANT_NAV_FOR_FINAL_PLOT = 1_000_000.0
UNIT_MAX_RISK_DOLLARS = 100_000.0

print("Current working directory:", cwd)
print("Project root:", PROJECT_ROOT)
print("Feature panel path:", FEATURE_PANEL_PATH)
print("Outcome path:", OUTCOME_PATH)
print("Feature panel exists:", FEATURE_PANEL_PATH.exists())
print("Outcome file exists:", OUTCOME_PATH.exists())
print("Model label:", MODEL_LABEL)
print("Win band:", WIN_BAND)


## Source files required

This notebook expects the following project files under `data/processed/`:

| Source file | Purpose |
|---|---|
| `production_feature_panel_v0_1.parquet` | Date × tenor feature panel containing SPX level, implied variance, forecast variance, VRP, rolling VRP z-scores, realized-volatility filter, and RSI filter. |
| `naked_atm_put_eod_panel_v0_1.parquet` | Completed naked ATM put outcome panel containing trade date, entry tenor, win/loss outcome, normalized P&L dollars, and actual DTE. |

The feature panel is the production feature set built from the SPX volatility surface and forecast/realized-volatility pipeline. The outcome panel is the completed EOD naked ATM put backtest panel used to evaluate selected tenors.


In [ ]:
# Cell 2 — Small reusable data-cleaning helpers

def get_col(df: pd.DataFrame, candidates, required=True, label=None):
    """
    Return the first matching column from a candidate list.

    Matching is case-insensitive, but the original DataFrame column name is
    returned. This keeps the notebook robust to small naming changes in source
    files.
    """
    lower_map = {str(c).lower(): c for c in df.columns}

    for c in candidates:
        if c in df.columns:
            return c

        c_lower = str(c).lower()
        if c_lower in lower_map:
            return lower_map[c_lower]

    if required:
        raise KeyError(
            f"Missing column for {label or candidates}. "
            f"Tried: {candidates}. Available columns: {list(df.columns)}"
        )

    return None


def parse_project_date_series(s: pd.Series) -> pd.Series:
    """
    Parse project date columns safely.

    Important:
      Numeric yyyymmdd values such as 20190805 must be parsed as calendar
      dates, not as nanoseconds since 1970.
    """
    s = s.copy()

    if pd.api.types.is_datetime64_any_dtype(s):
        return pd.to_datetime(s)

    s_str = s.astype("string").str.strip()
    s_str = s_str.str.replace(r"\.0$", "", regex=True)

    parsed = pd.Series(pd.NaT, index=s.index, dtype="datetime64[ns]")

    is_yyyymmdd = s_str.str.fullmatch(r"\d{8}", na=False)

    if is_yyyymmdd.any():
        parsed.loc[is_yyyymmdd] = pd.to_datetime(
            s_str.loc[is_yyyymmdd],
            format="%Y%m%d",
            errors="coerce",
        )

    remaining = ~is_yyyymmdd

    if remaining.any():
        parsed.loc[remaining] = pd.to_datetime(
            s_str.loc[remaining],
            errors="coerce",
        )

    return parsed


def clean_binary_series(s: pd.Series) -> pd.Series:
    """
    Convert bool / numeric / string binary values into float 0/1.
    Missing or unresolved values stay NaN.
    """
    if pd.api.types.is_bool_dtype(s):
        return s.astype(float)

    numeric = pd.to_numeric(s, errors="coerce")

    if numeric.notna().mean() > 0.95:
        return numeric.astype(float)

    return (
        s.astype("string")
        .str.lower()
        .str.strip()
        .map({
            "true": 1.0,
            "false": 0.0,
            "yes": 1.0,
            "no": 0.0,
            "1": 1.0,
            "0": 0.0,
            "1.0": 1.0,
            "0.0": 0.0,
        })
    )


def require_columns(df: pd.DataFrame, required_cols, label="DataFrame"):
    missing_cols = [c for c in required_cols if c not in df.columns]

    if missing_cols:
        raise ValueError(f"{label} is missing required columns: {missing_cols}")

In [ ]:
# Cell 3 — Load and validate the production feature panel

if not FEATURE_PANEL_PATH.exists():
    raise FileNotFoundError(f"Missing feature panel: {FEATURE_PANEL_PATH}")

features = pd.read_parquet(FEATURE_PANEL_PATH)
features["date"] = pd.to_datetime(features["date"]).dt.normalize()
features = features.sort_values(["date", "tenor"]).reset_index(drop=True)

required_feature_cols = [
    "date",
    "tenor",
    "tenor_group",
    "spx_close",
    "implied_variance",
    "forecast_variance",
    "vrp_log",
    "vrp_z_3m",
    "vrp_z_1y",
    "rv21d",
    "rsi14",
]

require_columns(features, required_feature_cols, label="Feature panel")

features = features[features["tenor"].isin(PRODUCTION_TENORS)].copy()
features["tenor"] = features["tenor"].astype(int)

eligible = features.dropna(
    subset=[
        "vrp_log",
        "vrp_z_3m",
        "vrp_z_1y",
        "rv21d",
        "rsi14",
        "implied_variance",
        "forecast_variance",
    ]
).copy()

eligible = eligible.sort_values(["date", "tenor"]).reset_index(drop=True)

dupes = eligible.duplicated(subset=["date", "tenor"]).sum()

print("Loaded feature panel")
print("  Shape:", features.shape)
print("  Date range:", features["date"].min(), "to", features["date"].max())
print("  Tenors:", sorted(features["tenor"].dropna().unique().tolist()))

print("\nEligible feature panel")
print("  Shape:", eligible.shape)
print("  Date range:", eligible["date"].min(), "to", eligible["date"].max())
print("  Tenors:", sorted(eligible["tenor"].dropna().unique().tolist()))
print("  Duplicate date/tenor rows:", dupes)

if dupes > 0:
    display(
        eligible[
            eligible.duplicated(subset=["date", "tenor"], keep=False)
        ].sort_values(["date", "tenor"]).head(30)
    )
    raise ValueError("Duplicate date/tenor rows found in eligible feature panel.")

# rv21d and rsi14 are market-level filters, so they should be identical
# across all tenors on a given date.
market_filter_counts = (
    eligible
    .groupby("date")[["rv21d", "rsi14"]]
    .nunique(dropna=False)
)

bad_market_filter_dates = market_filter_counts[
    (market_filter_counts["rv21d"] > 1) |
    (market_filter_counts["rsi14"] > 1)
]

print("  Dates with inconsistent rv21d/rsi14 across tenors:", len(bad_market_filter_dates))

if len(bad_market_filter_dates) > 0:
    display(bad_market_filter_dates.head(20))
    raise ValueError("rv21d or rsi14 differs across tenors on the same date.")

display(eligible.head(10))

In [ ]:
# Cell 4 — Load naked ATM put outcomes and build the completed research panel

if not OUTCOME_PATH.exists():
    raise FileNotFoundError(f"Missing outcome file: {OUTCOME_PATH}")

outcomes_raw = pd.read_parquet(OUTCOME_PATH)

# ---------------------------------------------------------------------
# Column mapping
# ---------------------------------------------------------------------

date_col = get_col(
    outcomes_raw,
    ["trade_dt", "trade_date", "date", "timestamp", "datetime"],
    label="date",
)

tenor_col = get_col(
    outcomes_raw,
    ["entry_tenor", "tenor", "target_tenor", "actual_dte", "dte"],
    label="tenor",
)

win_col = get_col(
    outcomes_raw,
    ["win_clean", "win", "test_win"],
    label="win",
)

expired_otm_col = get_col(
    outcomes_raw,
    ["expired_otm_clean", "expired_otm"],
    required=False,
    label="expired_otm",
)

pnl_dollars_col = get_col(
    outcomes_raw,
    ["normalized_pnl_dollars", "test_pnl", "sized_pnl"],
    required=False,
    label="normalized_pnl_dollars",
)

pnl_pct_col = get_col(
    outcomes_raw,
    ["normalized_pnl_pct"],
    required=False,
    label="normalized_pnl_pct",
)

actual_dte_col = get_col(
    outcomes_raw,
    ["actual_dte"],
    required=False,
    label="actual_dte",
)

expiry_spx_col = get_col(
    outcomes_raw,
    ["expiry_spx_close"],
    required=False,
    label="expiry_spx_close",
)

entry_credit_col = get_col(
    outcomes_raw,
    ["entry_credit_points", "atm_put_mid"],
    required=False,
    label="entry_credit_points",
)

short_pnl_points_col = get_col(
    outcomes_raw,
    ["short_option_pnl_points"],
    required=False,
    label="short_option_pnl_points",
)

print("Outcome column mapping:")
print({
    "date_col": date_col,
    "tenor_col": tenor_col,
    "win_col": win_col,
    "expired_otm_col": expired_otm_col,
    "pnl_dollars_col": pnl_dollars_col,
    "pnl_pct_col": pnl_pct_col,
    "actual_dte_col": actual_dte_col,
    "expiry_spx_col": expiry_spx_col,
    "entry_credit_col": entry_credit_col,
    "short_pnl_points_col": short_pnl_points_col,
})

# ---------------------------------------------------------------------
# Standardize outcomes
# ---------------------------------------------------------------------

outcomes = pd.DataFrame({
    "date": parse_project_date_series(outcomes_raw[date_col]).dt.normalize(),
    "tenor": pd.to_numeric(outcomes_raw[tenor_col], errors="coerce").astype("Int64"),
    "win_clean": clean_binary_series(outcomes_raw[win_col]),
})

if expired_otm_col is not None:
    outcomes["expired_otm_clean"] = clean_binary_series(outcomes_raw[expired_otm_col])
else:
    outcomes["expired_otm_clean"] = np.nan

if pnl_dollars_col is not None:
    outcomes["normalized_pnl_dollars"] = pd.to_numeric(outcomes_raw[pnl_dollars_col], errors="coerce")
else:
    outcomes["normalized_pnl_dollars"] = np.nan

if pnl_pct_col is not None:
    outcomes["normalized_pnl_pct"] = pd.to_numeric(outcomes_raw[pnl_pct_col], errors="coerce")
else:
    outcomes["normalized_pnl_pct"] = np.nan

if actual_dte_col is not None:
    outcomes["actual_dte"] = pd.to_numeric(outcomes_raw[actual_dte_col], errors="coerce")
else:
    outcomes["actual_dte"] = np.nan

if expiry_spx_col is not None:
    outcomes["expiry_spx_close"] = pd.to_numeric(outcomes_raw[expiry_spx_col], errors="coerce")
else:
    outcomes["expiry_spx_close"] = np.nan

if entry_credit_col is not None:
    outcomes["entry_credit_points"] = pd.to_numeric(outcomes_raw[entry_credit_col], errors="coerce")
else:
    outcomes["entry_credit_points"] = np.nan

if short_pnl_points_col is not None:
    outcomes["short_option_pnl_points"] = pd.to_numeric(outcomes_raw[short_pnl_points_col], errors="coerce")
else:
    outcomes["short_option_pnl_points"] = np.nan

outcomes = outcomes[outcomes["tenor"].isin(PRODUCTION_TENORS)].copy()
outcomes = outcomes.dropna(subset=["date", "tenor"]).copy()
outcomes["tenor"] = outcomes["tenor"].astype(int)

dupes = outcomes.duplicated(subset=["date", "tenor"]).sum()

print("\nStandardized outcome panel")
print("  Shape:", outcomes.shape)
print("  Date range:", outcomes["date"].min(), "to", outcomes["date"].max())
print("  Tenors:", sorted(outcomes["tenor"].dropna().unique().tolist()))
print("  Duplicate date/tenor rows:", dupes)

if dupes > 0:
    display(
        outcomes[
            outcomes.duplicated(subset=["date", "tenor"], keep=False)
        ].sort_values(["date", "tenor"]).head(30)
    )
    raise ValueError("Outcome panel has duplicate date/tenor rows.")

# ---------------------------------------------------------------------
# Join outcomes to eligible features.
# ---------------------------------------------------------------------
#
# Missing outcomes near the right edge are expected because later-dated
# longer-tenor trades may not have expired yet. Those rows are dropped.
#
# Missing outcomes before the last completed outcome date for that tenor are
# treated as a data hole and raise an error.
# ---------------------------------------------------------------------

research_panel_raw = eligible.merge(
    outcomes,
    on=["date", "tenor"],
    how="left",
)

completed_outcomes = outcomes.dropna(subset=["win_clean"]).copy()

last_completed_date_by_tenor = (
    completed_outcomes
    .groupby("tenor")["date"]
    .max()
    .rename("last_completed_outcome_date")
    .reset_index()
)

missing_outcomes = (
    research_panel_raw[research_panel_raw["win_clean"].isna()]
    [["date", "tenor", "tenor_group", "spx_close", "vrp_log", "vrp_z_3m", "vrp_z_1y", "rv21d", "rsi14"]]
    .copy()
)

missing_outcomes = missing_outcomes.merge(
    last_completed_date_by_tenor,
    on="tenor",
    how="left",
)

unexpected_missing = missing_outcomes[
    missing_outcomes["date"] <= missing_outcomes["last_completed_outcome_date"]
].copy()

print("\nResearch panel after outcome join")
print("  Shape:", research_panel_raw.shape)
print("  Rows with missing win_clean:", int(research_panel_raw["win_clean"].isna().sum()))

print("\nLast completed outcome date by tenor:")
display(last_completed_date_by_tenor)

if len(missing_outcomes) > 0:
    missing_summary = (
        missing_outcomes
        .groupby("tenor")
        .agg(
            missing_rows=("date", "size"),
            first_missing_date=("date", "min"),
            last_missing_date=("date", "max"),
            last_completed_outcome_date=("last_completed_outcome_date", "max"),
        )
        .reset_index()
    )

    print("\nMissing outcome summary by tenor:")
    display(missing_summary)

if len(unexpected_missing) > 0:
    print("\nUnexpected missing outcomes found before/equal to last completed date for that tenor:")
    display(unexpected_missing.head(30))
    raise ValueError("Unexpected non-right-edge missing outcomes found.")

print("\nRight-edge censored rows to drop:", len(missing_outcomes))

research_panel = research_panel_raw.dropna(subset=["win_clean"]).copy()
research_panel = research_panel.sort_values(["date", "tenor"]).reset_index(drop=True)

required_outcome_cols = ["win_clean", "normalized_pnl_dollars", "actual_dte"]
require_columns(research_panel, required_outcome_cols, label="Completed research panel")

if research_panel[required_outcome_cols].isna().any().any():
    display(research_panel[research_panel[required_outcome_cols].isna().any(axis=1)].head(20))
    raise ValueError("Completed research panel has missing win/P&L/actual_dte values.")

print("\nFinal completed research panel")
print("  Shape:", research_panel.shape)
print("  Date range:", research_panel["date"].min(), "to", research_panel["date"].max())
print("  Tenors:", sorted(research_panel["tenor"].dropna().unique().tolist()))
print("  Overall win rate:", research_panel["win_clean"].mean())

outcome_by_tenor = (
    research_panel
    .groupby("tenor")
    .agg(
        rows=("win_clean", "size"),
        win_rate=("win_clean", "mean"),
        avg_pnl_dollars=("normalized_pnl_dollars", "mean"),
        avg_pnl_pct=("normalized_pnl_pct", "mean"),
        avg_actual_dte=("actual_dte", "mean"),
    )
    .reset_index()
)

print("\nOutcome summary by tenor:")
display(outcome_by_tenor)

In [ ]:
# Cell 5 — Build complete-date sweep panel and matrix representation

# ---------------------------------------------------------------------
# Outcome normalization
# ---------------------------------------------------------------------
# The selected-tenor rule uses conditional P&L/day as a tie-breaker. This is
# not the same as using raw average P&L, which mechanically favors longer DTE.
# ---------------------------------------------------------------------

research_panel = research_panel.copy()

with np.errstate(divide="ignore", invalid="ignore"):
    research_panel["outcome_pnl_per_day"] = (
        research_panel["normalized_pnl_dollars"] / research_panel["actual_dte"]
    )

research_panel.loc[
    ~np.isfinite(research_panel["outcome_pnl_per_day"]),
    "outcome_pnl_per_day",
] = np.nan

# ---------------------------------------------------------------------
# Sweep panel cleaning
# ---------------------------------------------------------------------
# Keep only the columns needed for the locked signal evaluation and future
# overlay/sizing work.
# ---------------------------------------------------------------------

sweep_panel = research_panel[
    [
        "date",
        "tenor",
        "tenor_group",
        "spx_close",
        "vrp_log",
        "vrp_z_3m",
        "vrp_z_1y",
        "rv21d",
        "rsi14",
        "win_clean",
        "normalized_pnl_dollars",
        "actual_dte",
        "outcome_pnl_per_day",
    ]
].copy()

numeric_cols = [
    "tenor",
    "spx_close",
    "vrp_log",
    "vrp_z_3m",
    "vrp_z_1y",
    "rv21d",
    "rsi14",
    "win_clean",
    "normalized_pnl_dollars",
    "actual_dte",
    "outcome_pnl_per_day",
]

for col in numeric_cols:
    sweep_panel[col] = pd.to_numeric(sweep_panel[col], errors="coerce")

before_drop = len(sweep_panel)

sweep_panel = sweep_panel.dropna(
    subset=[
        "date",
        "tenor",
        "tenor_group",
        "spx_close",
        "vrp_log",
        "vrp_z_3m",
        "vrp_z_1y",
        "rv21d",
        "rsi14",
        "win_clean",
        "normalized_pnl_dollars",
        "actual_dte",
        "outcome_pnl_per_day",
    ]
).copy()

after_drop = len(sweep_panel)

print("\nSweep panel row cleaning")
print("  Rows before drop:", before_drop)
print("  Rows after drop: ", after_drop)
print("  Rows dropped:     ", before_drop - after_drop)

dupes = sweep_panel.duplicated(subset=["date", "tenor"]).sum()
print("  Duplicate date/tenor rows:", dupes)

if dupes > 0:
    display(
        sweep_panel[
            sweep_panel.duplicated(subset=["date", "tenor"], keep=False)
        ].sort_values(["date", "tenor"]).head(30)
    )
    raise ValueError("Duplicate date/tenor rows in sweep_panel.")

# ---------------------------------------------------------------------
# Complete-date filter
# ---------------------------------------------------------------------
# Keep only dates where all 9 tenors have completed outcomes. This keeps trade
# frequency comparisons clean and prevents right-edge censoring from biasing
# tenor selection.
# ---------------------------------------------------------------------

REQUIRED_TENOR_COUNT = len(PRODUCTION_TENORS)

tenor_count_by_date = (
    sweep_panel
    .groupby("date")["tenor"]
    .nunique()
    .rename("tenor_count")
)

complete_dates = tenor_count_by_date[tenor_count_by_date == REQUIRED_TENOR_COUNT].index
incomplete_dates = tenor_count_by_date[tenor_count_by_date != REQUIRED_TENOR_COUNT]

print("\nComplete-date filter")
print("  Original sweep rows:", len(sweep_panel))
print("  Original sweep dates:", sweep_panel["date"].nunique())
print("  Complete dates:", len(complete_dates))
print("  Incomplete dates:", len(incomplete_dates))

if len(incomplete_dates) > 0:
    print("\nIncomplete date summary:")
    display(
        incomplete_dates
        .reset_index()
        .sort_values("date")
    )

sweep_panel = sweep_panel[sweep_panel["date"].isin(complete_dates)].copy()
sweep_panel = sweep_panel.sort_values(["date", "tenor"]).reset_index(drop=True)

sweep_dates = pd.Index(sorted(sweep_panel["date"].unique()))
num_sweep_dates = len(sweep_dates)
TRADE_FREQUENCY_DENOMINATOR = num_sweep_dates

print("\nClean sweep panel")
print("  Rows:", len(sweep_panel))
print("  Dates:", num_sweep_dates)
print("  Date range:", sweep_dates.min(), "to", sweep_dates.max())
print("  Trade frequency denominator:", TRADE_FREQUENCY_DENOMINATOR)

clean_tenor_count_by_date = sweep_panel.groupby("date")["tenor"].nunique()

if not (clean_tenor_count_by_date == REQUIRED_TENOR_COUNT).all():
    raise ValueError("Some remaining dates do not have all 9 tenors.")

print("\nRows by tenor group after complete-date filter:")
display(
    sweep_panel
    .groupby("tenor_group")
    .agg(
        rows=("win_clean", "size"),
        win_rate=("win_clean", "mean"),
        avg_vrp=("vrp_log", "mean"),
        avg_z3m=("vrp_z_3m", "mean"),
        avg_z1y=("vrp_z_1y", "mean"),
        avg_rv21d=("rv21d", "mean"),
        avg_rsi14=("rsi14", "mean"),
        avg_pnl_per_day=("outcome_pnl_per_day", "mean"),
        total_pnl_dollars=("normalized_pnl_dollars", "sum"),
        total_actual_dte=("actual_dte", "sum"),
    )
    .assign(
        aggregate_pnl_per_day=lambda x: x["total_pnl_dollars"] / x["total_actual_dte"]
    )
    .reset_index()
)

# ---------------------------------------------------------------------
# Matrix representation
# ---------------------------------------------------------------------
# All strategy logic below uses matrix arrays:
#   rows = dates
#   columns = tenors [9, 12, ..., 33]
# ---------------------------------------------------------------------

TENOR_ARRAY = np.array(PRODUCTION_TENORS, dtype=int)
TENOR_ORDER = TENOR_ARRAY.tolist()
TENOR_TO_COL = {int(t): i for i, t in enumerate(TENOR_ARRAY)}

GROUP_COLS = {
    group: [TENOR_TO_COL[t] for t in tenors]
    for group, tenors in GROUP_TENORS.items()
}


def pivot_to_matrix(value_col):
    mat_df = (
        sweep_panel
        .pivot(index="date", columns="tenor", values=value_col)
        .reindex(index=sweep_dates, columns=PRODUCTION_TENORS)
    )

    if mat_df.isna().any().any():
        bad = mat_df[mat_df.isna().any(axis=1)].head(10)
        display(bad)
        raise ValueError(f"Matrix for {value_col} contains missing values.")

    return mat_df.to_numpy()


spx_mat = pivot_to_matrix("spx_close")
spx_close_mat = spx_mat
vrp_mat = pivot_to_matrix("vrp_log")
z3m_mat = pivot_to_matrix("vrp_z_3m")
z1y_mat = pivot_to_matrix("vrp_z_1y")
win_mat = pivot_to_matrix("win_clean")
pnl_mat = pivot_to_matrix("normalized_pnl_dollars")
actual_dte_mat = pivot_to_matrix("actual_dte")
pnl_per_day_mat = pivot_to_matrix("outcome_pnl_per_day")

rv21d_by_date = (
    sweep_panel.groupby("date")["rv21d"].first().reindex(sweep_dates).to_numpy()
)

rsi_by_date = (
    sweep_panel.groupby("date")["rsi14"].first().reindex(sweep_dates).to_numpy()
)

print("\nMatrix build complete")
print("  Matrix shape:", vrp_mat.shape)
print("  Tenor order:", TENOR_ORDER)


## Locked model specification

### Tenor universe and buckets

| Bucket | Tenors |
|---|---|
| Front | 9, 12, 15 DTE |
| Middle | 18, 21, 24 DTE |
| Back | 27, 30, 33 DTE |

### Core thresholds

| Bucket | VRP log | 3m VRP z-score | 1y VRP z-score | RSI cap | RV21D floor |
|---|---:|---:|---:|---:|---:|
| Front | 0.60 | 0.55 | 0.65 | 70 | 8.5 |
| Middle | 0.65 | 0.75 | 0.65 | 68 | 8.5 |
| Back | 0.70 | 0.75 | 0.75 | 66 | 8.5 |

### Secondary thresholds

| Bucket | VRP log | 3m VRP z-score | 1y VRP z-score | RSI cap | RV21D floor |
|---|---:|---:|---:|---:|---:|
| Front | 0.60 | 0.50 | 0.40 | 74 | 6.5 |
| Middle | 0.60 | 0.50 | 0.50 | 70 | 6.5 |
| Back | 0.70 | 0.50 | 0.50 | 68 | 6.5 |

### Selection logic

1. Check Core first.
2. If at least one Core tenor qualifies, select among Core tenors only.
3. If no Core tenor qualifies, check Secondary.
4. If at least one Secondary tenor qualifies, select among Secondary tenors only.
5. Select at most one trade per date.
6. Within the active layer, rank qualifying tenors by layer-specific conditional win probability.
7. Keep tenors within 25 bps of the best conditional win probability.
8. Among that near-tie set, choose the tenor with the highest conditional average P&L/day.
9. Remaining ties use aggregate P&L/day, conditional win probability, then longer tenor.


In [ ]:
# Cell 6 — Final locked 2621 blended tenor-priority model
#
# Final model convention:
#   1. Thresholds are locked to 2621.
#   2. Core is checked first.
#   3. Secondary is checked only if no Core tenor qualifies.
#   4. Within the active layer:
#        a. Rank tenors by layer-specific 2621-conditional win probability.
#        b. Keep tenors within 25 bps of the best qualifying tenor's win probability.
#        c. Select the tenor with highest layer-specific 2621-conditional avg P&L/day.
#        d. Tie-break by aggregate P&L/day, then conditional win probability, then longer tenor.
#
# Why:
#   - Avoids raw P&L duration bias.
#   - Avoids unconditional tenor statistics.
#   - Avoids pure P&L/day selection taking over the model.
#   - Keeps win probability as the primary criterion.
#
# Expected tenor behavior from bakeoff:
#   - Same front/middle/back mix as win_only.
#   - Main change: inside the back bucket, 27D can beat 33D when their conditional
#     win probabilities are effectively tied and 27D has better P&L/day.

import numpy as np
import pandas as pd
import json
from pathlib import Path

# ---------------------------------------------------------------------
# Robust setup
# ---------------------------------------------------------------------

if "TENOR_ORDER" in globals():
    TENOR_ARRAY = np.array(TENOR_ORDER, dtype=int)

elif "sweep_panel" in globals() and "tenor" in sweep_panel.columns:
    TENOR_ARRAY = np.array(sorted(sweep_panel["tenor"].dropna().unique()), dtype=int)
    TENOR_ORDER = TENOR_ARRAY.tolist()

elif "research_panel" in globals() and "tenor" in research_panel.columns:
    TENOR_ARRAY = np.array(sorted(research_panel["tenor"].dropna().unique()), dtype=int)
    TENOR_ORDER = TENOR_ARRAY.tolist()

else:
    TENOR_ARRAY = np.array([9, 12, 15, 18, 21, 24, 27, 30, 33], dtype=int)
    TENOR_ORDER = TENOR_ARRAY.tolist()

TENOR_TO_COL = {int(t): i for i, t in enumerate(TENOR_ARRAY)}

required_objects = [
    "num_sweep_dates",
    "sweep_dates",
    "vrp_mat",
    "z3m_mat",
    "z1y_mat",
    "win_mat",
    "pnl_mat",
    "actual_dte_mat",
    "rv21d_by_date",
    "rsi_by_date",
    "TRADE_FREQUENCY_DENOMINATOR",
]

missing_objects = [name for name in required_objects if name not in globals()]

if missing_objects:
    raise NameError(
        "Missing required objects. Run the data-loading / matrix-construction cells first. "
        f"Missing: {missing_objects}"
    )

if "pnl_per_day_mat" not in globals():
    with np.errstate(divide="ignore", invalid="ignore"):
        pnl_per_day_mat = pnl_mat / actual_dte_mat
    pnl_per_day_mat[~np.isfinite(pnl_per_day_mat)] = np.nan

if "AUDIT_DIR" not in globals():
    AUDIT_DIR = Path("data/audit")
else:
    AUDIT_DIR = Path(AUDIT_DIR)

if "PROCESSED_DIR" not in globals():
    PROCESSED_DIR = Path("data/processed")
else:
    PROCESSED_DIR = Path(PROCESSED_DIR)

AUDIT_DIR.mkdir(parents=True, exist_ok=True)
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)

MODEL_LABEL = "locked_2621_win_band_25bps_conditional"
WIN_BAND = 0.0025

print("Final locked 2621 blended-priority model")
print("Model label:", MODEL_LABEL)
print("Tenors:", TENOR_ARRAY.tolist())
print("Win band:", WIN_BAND)


# ---------------------------------------------------------------------
# Locked 2621 thresholds
# ---------------------------------------------------------------------

LOCKED_2621_CORE = {
    "front": {
        "tenors": [9, 12, 15],
        "vrp": 0.60,
        "z3m": 0.55,
        "z1y": 0.65,
        "rsi_cap": 70,
        "rv21d_floor": 8.5,
    },
    "middle": {
        "tenors": [18, 21, 24],
        "vrp": 0.65,
        "z3m": 0.75,
        "z1y": 0.65,
        "rsi_cap": 68,
        "rv21d_floor": 8.5,
    },
    "back": {
        "tenors": [27, 30, 33],
        "vrp": 0.70,
        "z3m": 0.75,
        "z1y": 0.75,
        "rsi_cap": 66,
        "rv21d_floor": 8.5,
    },
}

LOCKED_2621_SECONDARY = {
    "front": {
        "tenors": [9, 12, 15],
        "vrp": 0.60,
        "z3m": 0.50,
        "z1y": 0.40,
        "rsi_cap": 74,
        "rv21d_floor": 6.5,
    },
    "middle": {
        "tenors": [18, 21, 24],
        "vrp": 0.60,
        "z3m": 0.50,
        "z1y": 0.50,
        "rsi_cap": 70,
        "rv21d_floor": 6.5,
    },
    "back": {
        "tenors": [27, 30, 33],
        "vrp": 0.70,
        "z3m": 0.50,
        "z1y": 0.50,
        "rsi_cap": 68,
        "rv21d_floor": 6.5,
    },
}


# ---------------------------------------------------------------------
# Helper functions
# ---------------------------------------------------------------------

def tenor_group_for_tenor(tenor):
    tenor = int(tenor)

    if tenor in [9, 12, 15]:
        return "front"
    elif tenor in [18, 21, 24]:
        return "middle"
    elif tenor in [27, 30, 33]:
        return "back"
    else:
        raise ValueError(f"Unexpected tenor: {tenor}")


def build_threshold_table(threshold_spec, layer):
    rows = []

    for group, params in threshold_spec.items():
        for tenor in params["tenors"]:
            rows.append({
                "layer": layer,
                "tenor_group": group,
                "selected_tenor": int(tenor),
                "vrp_threshold": float(params["vrp"]),
                "z3m_threshold": float(params["z3m"]),
                "z1y_threshold": float(params["z1y"]),
                "rsi_cap": float(params["rsi_cap"]),
                "rv21d_floor": float(params["rv21d_floor"]),
            })

    return pd.DataFrame(rows)


def build_qualifies_matrix(threshold_spec):
    """
    Builds date x tenor boolean qualification matrix for locked 2621 thresholds.
    """
    qualifies = np.zeros((num_sweep_dates, len(TENOR_ARRAY)), dtype=bool)

    for col, tenor in enumerate(TENOR_ARRAY):
        group = tenor_group_for_tenor(tenor)
        params = threshold_spec[group]

        qualifies[:, col] = (
            (vrp_mat[:, col] >= float(params["vrp"])) &
            (z3m_mat[:, col] >= float(params["z3m"])) &
            (z1y_mat[:, col] >= float(params["z1y"])) &
            (rsi_by_date <= float(params["rsi_cap"])) &
            (rv21d_by_date >= float(params["rv21d_floor"]))
        )

    return qualifies


def build_layer_conditional_tenor_metrics(layer_name, qualifies_matrix, date_mask):
    """
    Builds tenor ranking metrics conditional on the relevant 2621 candidate universe.

    Core:
      date_mask = all dates.

    Secondary:
      date_mask = dates where no Core tenor qualifies.
    """
    rows = []

    for col, tenor in enumerate(TENOR_ARRAY):
        candidate_mask = (
            date_mask &
            qualifies_matrix[:, col] &
            np.isfinite(win_mat[:, col]) &
            np.isfinite(pnl_mat[:, col]) &
            np.isfinite(actual_dte_mat[:, col]) &
            np.isfinite(pnl_per_day_mat[:, col])
        )

        candidate_count = int(candidate_mask.sum())

        if candidate_count == 0:
            rows.append({
                "layer": layer_name,
                "selected_col": int(col),
                "selected_tenor": int(tenor),
                "tenor_group": tenor_group_for_tenor(tenor),
                "conditional_win_probability": np.nan,
                "conditional_avg_pnl_per_day": np.nan,
                "conditional_median_pnl_per_day": np.nan,
                "conditional_aggregate_pnl_per_day": np.nan,
                "conditional_avg_raw_pnl": np.nan,
                "conditional_avg_actual_dte": np.nan,
                "conditional_worst_trade_pnl": np.nan,
                "conditional_worst_pnl_per_day": np.nan,
                "candidate_count": 0,
            })
            continue

        rows.append({
            "layer": layer_name,
            "selected_col": int(col),
            "selected_tenor": int(tenor),
            "tenor_group": tenor_group_for_tenor(tenor),

            "conditional_win_probability": float(np.nanmean(win_mat[candidate_mask, col])),
            "conditional_avg_pnl_per_day": float(np.nanmean(pnl_per_day_mat[candidate_mask, col])),
            "conditional_median_pnl_per_day": float(np.nanmedian(pnl_per_day_mat[candidate_mask, col])),
            "conditional_aggregate_pnl_per_day": float(
                np.nansum(pnl_mat[candidate_mask, col]) /
                np.nansum(actual_dte_mat[candidate_mask, col])
            ),
            "conditional_avg_raw_pnl": float(np.nanmean(pnl_mat[candidate_mask, col])),
            "conditional_avg_actual_dte": float(np.nanmean(actual_dte_mat[candidate_mask, col])),
            "conditional_worst_trade_pnl": float(np.nanmin(pnl_mat[candidate_mask, col])),
            "conditional_worst_pnl_per_day": float(np.nanmin(pnl_per_day_mat[candidate_mask, col])),
            "candidate_count": candidate_count,
        })

    return pd.DataFrame(rows)


def blended_priority_order(layer_name, qualifying_cols, win_band=WIN_BAND):
    """
    Final blended priority rule.

    Step 1:
      Find best conditional win probability among qualifying tenors.

    Step 2:
      Keep tenors within win_band of that best conditional win probability.

    Step 3:
      Select highest conditional avg P&L/day.
      Tie-break by aggregate P&L/day, then conditional win probability, then longer tenor.
    """
    if len(qualifying_cols) == 0:
        return pd.DataFrame()

    df = tenor_priority_metrics[
        (tenor_priority_metrics["layer"].eq(layer_name)) &
        (tenor_priority_metrics["selected_col"].isin(qualifying_cols)) &
        (tenor_priority_metrics["candidate_count"] > 0)
    ].copy()

    if df.empty:
        return df

    best_win = df["conditional_win_probability"].max()

    df["best_conditional_win_probability"] = best_win
    df["win_probability_gap_to_best"] = best_win - df["conditional_win_probability"]
    df["inside_win_band"] = df["win_probability_gap_to_best"] <= win_band

    df = df[df["inside_win_band"]].copy()

    df = df.sort_values(
        [
            "conditional_avg_pnl_per_day",
            "conditional_aggregate_pnl_per_day",
            "conditional_win_probability",
            "selected_tenor",
        ],
        ascending=[False, False, False, False],
    ).reset_index(drop=True)

    df["priority_rank_within_signal"] = np.arange(1, len(df) + 1)

    return df


def select_final_model_path():
    """
    Core-first / Secondary-second final path selection.
    """
    selected_rows = []

    core_selected = np.full(num_sweep_dates, -1, dtype=np.int8)
    secondary_selected = np.full(num_sweep_dates, -1, dtype=np.int8)
    combined_selected = np.full(num_sweep_dates, -1, dtype=np.int8)

    for row in range(num_sweep_dates):
        core_qualifying_cols = np.where(core_qualifies_matrix[row, :])[0].astype(int).tolist()

        if core_qualifying_cols:
            ordered = blended_priority_order(
                layer_name="Core",
                qualifying_cols=core_qualifying_cols,
                win_band=WIN_BAND,
            )

            if not ordered.empty:
                selected_col = int(ordered.iloc[0]["selected_col"])

                core_selected[row] = selected_col
                combined_selected[row] = selected_col

                selected_rows.append({
                    "row_idx": int(row),
                    "date": sweep_dates[row],
                    "layer": "Core",
                    "selected_col": selected_col,
                    "selected_tenor": int(TENOR_ARRAY[selected_col]),
                    "tenor_group": tenor_group_for_tenor(TENOR_ARRAY[selected_col]),
                    "num_qualifying_tenors_in_layer": int(len(core_qualifying_cols)),
                    "qualifying_tenors_in_layer": ",".join(
                        [str(int(TENOR_ARRAY[c])) for c in core_qualifying_cols]
                    ),
                    "selection_rule": "win_band_25bps_then_pnl_day",
                    "win_band": WIN_BAND,
                })

            continue

        secondary_qualifying_cols = np.where(secondary_qualifies_matrix[row, :])[0].astype(int).tolist()

        if secondary_qualifying_cols:
            ordered = blended_priority_order(
                layer_name="Secondary",
                qualifying_cols=secondary_qualifying_cols,
                win_band=WIN_BAND,
            )

            if not ordered.empty:
                selected_col = int(ordered.iloc[0]["selected_col"])

                secondary_selected[row] = selected_col
                combined_selected[row] = selected_col

                selected_rows.append({
                    "row_idx": int(row),
                    "date": sweep_dates[row],
                    "layer": "Secondary",
                    "selected_col": selected_col,
                    "selected_tenor": int(TENOR_ARRAY[selected_col]),
                    "tenor_group": tenor_group_for_tenor(TENOR_ARRAY[selected_col]),
                    "num_qualifying_tenors_in_layer": int(len(secondary_qualifying_cols)),
                    "qualifying_tenors_in_layer": ",".join(
                        [str(int(TENOR_ARRAY[c])) for c in secondary_qualifying_cols]
                    ),
                    "selection_rule": "win_band_25bps_then_pnl_day",
                    "win_band": WIN_BAND,
                })

    selected_path = pd.DataFrame(selected_rows)

    return selected_path, {
        "core_selected": core_selected,
        "secondary_selected": secondary_selected,
        "combined_selected": combined_selected,
    }


def attach_trade_outcomes(selected_path):
    """
    Adds outcomes, features, and conditional ranking metrics to selected path.
    """
    if selected_path.empty:
        return selected_path.copy()

    row_idx = selected_path["row_idx"].to_numpy(dtype=int)
    col_idx = selected_path["selected_col"].to_numpy(dtype=int)

    trades = selected_path.copy()

    trades["candidate"] = MODEL_LABEL

    trades["win"] = win_mat[row_idx, col_idx]
    trades["normalized_pnl_dollars"] = pnl_mat[row_idx, col_idx]
    trades["actual_dte"] = actual_dte_mat[row_idx, col_idx]
    trades["pnl_per_day"] = pnl_per_day_mat[row_idx, col_idx]

    trades["vrp_log"] = vrp_mat[row_idx, col_idx]
    trades["vrp_z_3m"] = z3m_mat[row_idx, col_idx]
    trades["vrp_z_1y"] = z1y_mat[row_idx, col_idx]
    trades["rv21d"] = rv21d_by_date[row_idx]
    trades["rsi14"] = rsi_by_date[row_idx]

    if "spx_close_mat" in globals():
        trades["spx_close"] = spx_close_mat[row_idx, col_idx]
    elif "spx_close_by_date" in globals():
        trades["spx_close"] = np.asarray(spx_close_by_date)[row_idx]
    else:
        trades["spx_close"] = np.nan

    trades = trades.merge(
        tenor_priority_metrics,
        on=["layer", "selected_col", "selected_tenor", "tenor_group"],
        how="left",
    )

    trades = trades.sort_values("date").reset_index(drop=True)

    trades["trade_sequence"] = np.arange(1, len(trades) + 1)
    trades["cum_pnl"] = trades["normalized_pnl_dollars"].cumsum()
    trades["running_max_cum_pnl"] = trades["cum_pnl"].cummax()
    trades["drawdown"] = trades["cum_pnl"] - trades["running_max_cum_pnl"]
    trades["rolling_20_trade_pnl"] = trades["normalized_pnl_dollars"].rolling(20).sum()

    trades["year"] = pd.to_datetime(trades["date"]).dt.year
    trades["month"] = pd.to_datetime(trades["date"]).dt.to_period("M").astype(str)

    return trades


def summarize_trade_df(df, label):
    """
    Summary for one path or subgroup.
    """
    trade_count = int(len(df))

    if trade_count == 0:
        return pd.Series({
            "label": label,
            "trade_count": 0,
            "trade_frequency": 0.0,
            "win_rate": np.nan,
            "avg_pnl_per_day": np.nan,
            "median_pnl_per_day": np.nan,
            "aggregate_pnl_per_day": np.nan,
            "total_pnl": 0.0,
            "total_actual_dte": 0.0,
            "worst_trade_pnl": np.nan,
            "worst_pnl_per_day": np.nan,
            "max_drawdown": np.nan,
            "worst_20_trade_sum": np.nan,
            "avg_selected_tenor": np.nan,
        })

    df = df.sort_values("date").copy()

    df["local_cum_pnl"] = df["normalized_pnl_dollars"].cumsum()
    df["local_running_max"] = df["local_cum_pnl"].cummax()
    df["local_drawdown"] = df["local_cum_pnl"] - df["local_running_max"]
    df["local_rolling_20_trade_pnl"] = df["normalized_pnl_dollars"].rolling(20).sum()

    total_pnl = float(df["normalized_pnl_dollars"].sum())
    total_actual_dte = float(df["actual_dte"].sum())

    return pd.Series({
        "label": label,
        "trade_count": trade_count,
        "trade_frequency": float(trade_count / TRADE_FREQUENCY_DENOMINATOR),
        "win_rate": float(df["win"].mean()),
        "avg_pnl_per_day": float(df["pnl_per_day"].mean()),
        "median_pnl_per_day": float(df["pnl_per_day"].median()),
        "aggregate_pnl_per_day": float(total_pnl / total_actual_dte) if total_actual_dte else np.nan,
        "total_pnl": total_pnl,
        "total_actual_dte": total_actual_dte,
        "worst_trade_pnl": float(df["normalized_pnl_dollars"].min()),
        "worst_pnl_per_day": float(df["pnl_per_day"].min()),
        "max_drawdown": float(df["local_drawdown"].min()),
        "worst_20_trade_sum": float(df["local_rolling_20_trade_pnl"].min()),
        "avg_selected_tenor": float(df["selected_tenor"].mean()),
    })


def summarize_by_group(df, group_cols):
    rows = []

    for keys, sub in df.groupby(group_cols, dropna=False):
        if not isinstance(keys, tuple):
            keys = (keys,)

        row = {col: key for col, key in zip(group_cols, keys)}
        summary = summarize_trade_df(sub, label="group").to_dict()
        summary.pop("label", None)

        row.update(summary)
        rows.append(row)

    return pd.DataFrame(rows)


# ---------------------------------------------------------------------
# Build qualification matrices and conditional tenor metrics
# ---------------------------------------------------------------------

core_qualifies_matrix = build_qualifies_matrix(LOCKED_2621_CORE)
secondary_qualifies_matrix = build_qualifies_matrix(LOCKED_2621_SECONDARY)

any_core_qualifies_by_date = core_qualifies_matrix.any(axis=1)
no_core_qualifies_by_date = ~any_core_qualifies_by_date

core_tenor_priority_metrics = build_layer_conditional_tenor_metrics(
    layer_name="Core",
    qualifies_matrix=core_qualifies_matrix,
    date_mask=np.ones(num_sweep_dates, dtype=bool),
)

secondary_tenor_priority_metrics = build_layer_conditional_tenor_metrics(
    layer_name="Secondary",
    qualifies_matrix=secondary_qualifies_matrix,
    date_mask=no_core_qualifies_by_date,
)

tenor_priority_metrics = pd.concat(
    [core_tenor_priority_metrics, secondary_tenor_priority_metrics],
    ignore_index=True,
)

locked_2621_threshold_table = pd.concat(
    [
        build_threshold_table(LOCKED_2621_CORE, "Core"),
        build_threshold_table(LOCKED_2621_SECONDARY, "Secondary"),
    ],
    ignore_index=True,
)

print("\nLocked 2621 qualification counts:")
print("Dates with at least one Core tenor:", int(any_core_qualifies_by_date.sum()))
print("Dates with no Core tenor:", int(no_core_qualifies_by_date.sum()))
print("Dates with at least one Secondary tenor:", int(secondary_qualifies_matrix.any(axis=1).sum()))
print(
    "Dates with at least one Secondary tenor and no Core tenor:",
    int((secondary_qualifies_matrix.any(axis=1) & no_core_qualifies_by_date).sum()),
)

print("\nCore 2621-conditional tenor priority metrics:")
display(
    core_tenor_priority_metrics.sort_values(
        ["conditional_win_probability", "conditional_avg_pnl_per_day"],
        ascending=[False, False],
    )
)

print("\nSecondary 2621-conditional tenor priority metrics:")
display(
    secondary_tenor_priority_metrics.sort_values(
        ["conditional_win_probability", "conditional_avg_pnl_per_day"],
        ascending=[False, False],
    )
)


# ---------------------------------------------------------------------
# Select final model path
# ---------------------------------------------------------------------

selected_path, selected_arrays = select_final_model_path()
locked_2621_selected_trades = attach_trade_outcomes(selected_path)

locked_2621_path_summary = pd.DataFrame(
    [summarize_trade_df(locked_2621_selected_trades, label="Combined")]
)

summary_by_layer = summarize_by_group(
    locked_2621_selected_trades,
    ["layer"],
)

summary_by_year_layer = summarize_by_group(
    locked_2621_selected_trades,
    ["year", "layer"],
)

summary_by_tenor_layer = summarize_by_group(
    locked_2621_selected_trades,
    ["selected_tenor", "layer"],
)

summary_by_group_layer = summarize_by_group(
    locked_2621_selected_trades,
    ["tenor_group", "layer"],
)

worst_trades = (
    locked_2621_selected_trades
    .sort_values("normalized_pnl_dollars", ascending=True)
    .head(20)
    .reset_index(drop=True)
)

worst_20_trade_windows = (
    locked_2621_selected_trades
    .dropna(subset=["rolling_20_trade_pnl"])
    .sort_values("rolling_20_trade_pnl", ascending=True)
    .head(20)
    .reset_index(drop=True)
)

print("\nFinal locked 2621 blended-priority path summary:")
display(locked_2621_path_summary)

print("\nSummary by layer:")
display(summary_by_layer)

print("\nSummary by year and layer:")
display(summary_by_year_layer)

print("\nSummary by tenor and layer:")
display(summary_by_tenor_layer)

print("\nSummary by tenor group and layer:")
display(summary_by_group_layer)

print("\nWorst 20 individual trades:")
display(worst_trades)

print("\nWorst 20-trade rolling windows:")
display(worst_20_trade_windows)


# ---------------------------------------------------------------------
# Save final model outputs
# ---------------------------------------------------------------------

SELECTED_TRADES_CSV_PATH = AUDIT_DIR / f"{MODEL_LABEL}_selected_trades_v0_1.csv"
SELECTED_TRADES_PARQUET_PATH = PROCESSED_DIR / f"{MODEL_LABEL}_selected_trades_v0_1.parquet"

SUMMARY_PATH = AUDIT_DIR / f"{MODEL_LABEL}_summary_v0_1.csv"
SUMMARY_BY_LAYER_PATH = AUDIT_DIR / f"{MODEL_LABEL}_summary_by_layer_v0_1.csv"
SUMMARY_BY_YEAR_LAYER_PATH = AUDIT_DIR / f"{MODEL_LABEL}_summary_by_year_layer_v0_1.csv"
SUMMARY_BY_TENOR_LAYER_PATH = AUDIT_DIR / f"{MODEL_LABEL}_summary_by_tenor_layer_v0_1.csv"
SUMMARY_BY_GROUP_LAYER_PATH = AUDIT_DIR / f"{MODEL_LABEL}_summary_by_group_layer_v0_1.csv"

WORST_TRADES_PATH = AUDIT_DIR / f"{MODEL_LABEL}_worst_trades_v0_1.csv"
WORST_WINDOWS_PATH = AUDIT_DIR / f"{MODEL_LABEL}_worst_20_trade_windows_v0_1.csv"

TENOR_PRIORITY_METRICS_PATH = AUDIT_DIR / f"{MODEL_LABEL}_tenor_priority_metrics_v0_1.csv"
THRESHOLD_TABLE_PATH = AUDIT_DIR / f"{MODEL_LABEL}_thresholds_v0_1.csv"
SUMMARY_JSON_PATH = AUDIT_DIR / f"{MODEL_LABEL}_summary_v0_1.json"

locked_2621_selected_trades.to_csv(SELECTED_TRADES_CSV_PATH, index=False)
locked_2621_selected_trades.to_parquet(SELECTED_TRADES_PARQUET_PATH, index=False)

locked_2621_path_summary.to_csv(SUMMARY_PATH, index=False)
summary_by_layer.to_csv(SUMMARY_BY_LAYER_PATH, index=False)
summary_by_year_layer.to_csv(SUMMARY_BY_YEAR_LAYER_PATH, index=False)
summary_by_tenor_layer.to_csv(SUMMARY_BY_TENOR_LAYER_PATH, index=False)
summary_by_group_layer.to_csv(SUMMARY_BY_GROUP_LAYER_PATH, index=False)

worst_trades.to_csv(WORST_TRADES_PATH, index=False)
worst_20_trade_windows.to_csv(WORST_WINDOWS_PATH, index=False)

tenor_priority_metrics.to_csv(TENOR_PRIORITY_METRICS_PATH, index=False)
locked_2621_threshold_table.to_csv(THRESHOLD_TABLE_PATH, index=False)

summary_json = {
    "model_label": MODEL_LABEL,
    "thresholds": "locked_2621",
    "priority_rule": "layer-specific 2621-conditional win probability within 25 bps, then conditional avg P&L/day",
    "win_band": WIN_BAND,
    "trade_count": int(locked_2621_path_summary.iloc[0]["trade_count"]),
    "trade_frequency": float(locked_2621_path_summary.iloc[0]["trade_frequency"]),
    "win_rate": float(locked_2621_path_summary.iloc[0]["win_rate"]),
    "avg_pnl_per_day": float(locked_2621_path_summary.iloc[0]["avg_pnl_per_day"]),
    "aggregate_pnl_per_day": float(locked_2621_path_summary.iloc[0]["aggregate_pnl_per_day"]),
    "total_pnl": float(locked_2621_path_summary.iloc[0]["total_pnl"]),
    "max_drawdown": float(locked_2621_path_summary.iloc[0]["max_drawdown"]),
    "worst_20_trade_sum": float(locked_2621_path_summary.iloc[0]["worst_20_trade_sum"]),
    "avg_selected_tenor": float(locked_2621_path_summary.iloc[0]["avg_selected_tenor"]),
    "selected_trades_csv": str(SELECTED_TRADES_CSV_PATH),
    "selected_trades_parquet": str(SELECTED_TRADES_PARQUET_PATH),
}

with open(SUMMARY_JSON_PATH, "w") as f:
    json.dump(summary_json, f, indent=2)

print("\nSaved final locked 2621 blended-priority outputs:")
print(SELECTED_TRADES_CSV_PATH)
print(SELECTED_TRADES_PARQUET_PATH)
print(SUMMARY_PATH)
print(SUMMARY_BY_LAYER_PATH)
print(SUMMARY_BY_YEAR_LAYER_PATH)
print(SUMMARY_BY_TENOR_LAYER_PATH)
print(SUMMARY_BY_GROUP_LAYER_PATH)
print(WORST_TRADES_PATH)
print(WORST_WINDOWS_PATH)
print(TENOR_PRIORITY_METRICS_PATH)
print(THRESHOLD_TABLE_PATH)
print(SUMMARY_JSON_PATH)

print("\nFinal locked 2621 blended-priority model complete.")

## Locked per-trade max-risk sizing

Sizing is defined as theoretical max loss / max risk at inception as a percentage of the constant sizing NAV.

| Group | Max-risk fraction |
|---|---:|
| Core_back | 5.00% |
| Core_middle | 4.85% |
| Core_front | 1.75% |
| Secondary_back | 4.50% |
| Secondary_middle | 3.25% |
| Secondary_front | 1.75% |

For the final plot, every trade is sized against a constant $1,000,000 NAV. There is no account growth, no account loss, and no compounding.


In [ ]:
# Cell 7 — Apply locked sizing and plot cumulative P&L with constant $1mm NAV
#
# Important:
#   - This does NOT compound.
#   - Every trade is sized as if NAV is always exactly $1,000,000.
#   - Portfolio overlap, hedge logic, and rolling exposure caps are intentionally excluded.

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path

CONSTANT_NAV = CONSTANT_NAV_FOR_FINAL_PLOT
PLOT_LABEL = "locked_2621_constant_1mm_nav_cumulative_pnl"

print("Plot label:", PLOT_LABEL)
print("Constant NAV sizing base:", CONSTANT_NAV)
print("Unit max-risk dollars:", UNIT_MAX_RISK_DOLLARS)
print("Locked per-trade risk map:")
print(LOCKED_PER_TRADE_RISK_FRACTION)

# ---------------------------------------------------------------------
# Load selected trades from memory or disk
# ---------------------------------------------------------------------

if "locked_2621_selected_trades" in globals():
    plot_trades = locked_2621_selected_trades.copy()
else:
    selected_trades_path = PROCESSED_DIR / f"{MODEL_LABEL}_selected_trades_v0_1.parquet"
    if not selected_trades_path.exists():
        raise FileNotFoundError(
            "Selected trades not found in memory or on disk. "
            "Run the locked model selection cell first. "
            f"Expected: {selected_trades_path}"
        )
    plot_trades = pd.read_parquet(selected_trades_path)

plot_trades = plot_trades.loc[:, ~plot_trades.columns.duplicated()].copy()

required_cols = [
    "date",
    "layer",
    "tenor_group",
    "selected_tenor",
    "actual_dte",
    "win",
    "normalized_pnl_dollars",
]

missing_cols = [c for c in required_cols if c not in plot_trades.columns]
if missing_cols:
    raise ValueError(f"Missing required columns: {missing_cols}")

plot_trades["date"] = pd.to_datetime(plot_trades["date"]).dt.normalize()
plot_trades["actual_dte"] = pd.to_numeric(plot_trades["actual_dte"], errors="coerce")
plot_trades["normalized_pnl_dollars"] = pd.to_numeric(
    plot_trades["normalized_pnl_dollars"],
    errors="coerce",
)

# If an explicit expiry/exit date is unavailable, approximate it with actual DTE.
if "expiration_date" in plot_trades.columns:
    plot_trades["exit_date"] = pd.to_datetime(plot_trades["expiration_date"]).dt.normalize()
elif "expiry_date" in plot_trades.columns:
    plot_trades["exit_date"] = pd.to_datetime(plot_trades["expiry_date"]).dt.normalize()
elif "exit_date" in plot_trades.columns:
    plot_trades["exit_date"] = pd.to_datetime(plot_trades["exit_date"]).dt.normalize()
else:
    plot_trades["exit_date"] = (
        plot_trades["date"]
        + pd.to_timedelta(np.ceil(plot_trades["actual_dte"]).astype(int), unit="D")
    )

# ---------------------------------------------------------------------
# Apply final locked sizing
# ---------------------------------------------------------------------

def get_final_sizing_group(row):
    return f"{row['layer']}_{row['tenor_group']}"

plot_trades["final_sizing_group"] = plot_trades.apply(get_final_sizing_group, axis=1)

unknown_groups = sorted(
    set(plot_trades["final_sizing_group"]) - set(LOCKED_PER_TRADE_RISK_FRACTION.keys())
)
if unknown_groups:
    raise ValueError(f"Unexpected final_sizing_group values: {unknown_groups}")

plot_trades["final_risk_fraction"] = plot_trades["final_sizing_group"].map(
    LOCKED_PER_TRADE_RISK_FRACTION
)

plot_trades["constant_nav_trade_max_risk_dollars"] = (
    CONSTANT_NAV * plot_trades["final_risk_fraction"]
)
plot_trades["constant_nav_unit_count"] = (
    plot_trades["constant_nav_trade_max_risk_dollars"] / UNIT_MAX_RISK_DOLLARS
)
plot_trades["constant_nav_sized_pnl_dollars"] = (
    plot_trades["constant_nav_unit_count"]
    * plot_trades["normalized_pnl_dollars"]
)
plot_trades["constant_nav_return_on_nav"] = (
    plot_trades["constant_nav_sized_pnl_dollars"] / CONSTANT_NAV
)

# ---------------------------------------------------------------------
# Realized P&L curve by close / exit date
# ---------------------------------------------------------------------

realized_curve = (
    plot_trades
    .sort_values(["exit_date", "date", "selected_tenor"])
    .reset_index(drop=True)
    .copy()
)

realized_curve["trade_number"] = np.arange(1, len(realized_curve) + 1)
realized_curve["cumulative_pnl_dollars"] = realized_curve["constant_nav_sized_pnl_dollars"].cumsum()
realized_curve["cumulative_return_on_nav"] = realized_curve["cumulative_pnl_dollars"] / CONSTANT_NAV
realized_curve["running_peak_pnl"] = realized_curve["cumulative_pnl_dollars"].cummax()
realized_curve["drawdown_dollars"] = realized_curve["cumulative_pnl_dollars"] - realized_curve["running_peak_pnl"]
realized_curve["drawdown_pct_of_nav"] = realized_curve["drawdown_dollars"] / CONSTANT_NAV
realized_curve["rolling_20_trade_pnl"] = realized_curve["constant_nav_sized_pnl_dollars"].rolling(20).sum()

summary = {
    "trades": int(len(realized_curve)),
    "start_entry_date": plot_trades["date"].min(),
    "last_entry_date": plot_trades["date"].max(),
    "first_exit_date": realized_curve["exit_date"].min(),
    "last_exit_date": realized_curve["exit_date"].max(),
    "constant_nav": CONSTANT_NAV,
    "total_pnl_dollars": float(realized_curve["cumulative_pnl_dollars"].iloc[-1]),
    "total_return_on_constant_nav": float(realized_curve["cumulative_return_on_nav"].iloc[-1]),
    "win_rate": float((plot_trades["constant_nav_sized_pnl_dollars"] > 0).mean()),
    "avg_trade_pnl_dollars": float(plot_trades["constant_nav_sized_pnl_dollars"].mean()),
    "median_trade_pnl_dollars": float(plot_trades["constant_nav_sized_pnl_dollars"].median()),
    "worst_trade_pnl_dollars": float(plot_trades["constant_nav_sized_pnl_dollars"].min()),
    "best_trade_pnl_dollars": float(plot_trades["constant_nav_sized_pnl_dollars"].max()),
    "max_drawdown_dollars": float(realized_curve["drawdown_dollars"].min()),
    "max_drawdown_pct_of_nav": float(realized_curve["drawdown_pct_of_nav"].min()),
    "worst_20_trade_sum_dollars": float(realized_curve["rolling_20_trade_pnl"].min()),
}

summary_df = pd.DataFrame([summary])

print("\nConstant $1mm NAV summary:")
display(summary_df)

print("\nP&L by final sizing group:")
display(
    plot_trades
    .groupby("final_sizing_group", as_index=False)
    .agg(
        trades=("date", "count"),
        risk_fraction=("final_risk_fraction", "first"),
        total_max_risk_dollars=("constant_nav_trade_max_risk_dollars", "sum"),
        total_pnl_dollars=("constant_nav_sized_pnl_dollars", "sum"),
        avg_trade_pnl_dollars=("constant_nav_sized_pnl_dollars", "mean"),
        median_trade_pnl_dollars=("constant_nav_sized_pnl_dollars", "median"),
        win_rate=("constant_nav_sized_pnl_dollars", lambda x: float((x > 0).mean())),
        worst_trade_pnl_dollars=("constant_nav_sized_pnl_dollars", "min"),
        best_trade_pnl_dollars=("constant_nav_sized_pnl_dollars", "max"),
    )
    .sort_values("total_pnl_dollars", ascending=False)
)

print("\nP&L by year:")
display(
    plot_trades
    .assign(year=lambda x: x["date"].dt.year)
    .groupby("year", as_index=False)
    .agg(
        trades=("date", "count"),
        total_pnl_dollars=("constant_nav_sized_pnl_dollars", "sum"),
        avg_trade_pnl_dollars=("constant_nav_sized_pnl_dollars", "mean"),
        win_rate=("constant_nav_sized_pnl_dollars", lambda x: float((x > 0).mean())),
        worst_trade_pnl_dollars=("constant_nav_sized_pnl_dollars", "min"),
    )
)

# ---------------------------------------------------------------------
# Plot
# ---------------------------------------------------------------------

fig, ax = plt.subplots(figsize=(14, 7))

ax.plot(
    realized_curve["exit_date"],
    realized_curve["cumulative_pnl_dollars"],
    linewidth=2,
)
ax.axhline(0, linewidth=1)
ax.set_title("Cumulative realized P&L — locked sizing, constant $1mm NAV")
ax.set_xlabel("Exit date")
ax.set_ylabel("Cumulative P&L dollars")
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

# ---------------------------------------------------------------------
# Save outputs
# ---------------------------------------------------------------------

REALIZED_CURVE_PATH = AUDIT_DIR / f"{PLOT_LABEL}_realized_curve_v0_1.csv"
TRADE_LEVEL_PATH = AUDIT_DIR / f"{PLOT_LABEL}_trade_level_v0_1.csv"
SUMMARY_PATH = AUDIT_DIR / f"{PLOT_LABEL}_summary_v0_1.csv"
PNG_PATH = AUDIT_DIR / f"{PLOT_LABEL}_cumulative_pnl_v0_1.png"

realized_curve.to_csv(REALIZED_CURVE_PATH, index=False)
plot_trades.to_csv(TRADE_LEVEL_PATH, index=False)
summary_df.to_csv(SUMMARY_PATH, index=False)
fig.savefig(PNG_PATH, dpi=150, bbox_inches="tight")

print("\nSaved outputs:")
print(REALIZED_CURVE_PATH)
print(TRADE_LEVEL_PATH)
print(SUMMARY_PATH)
print(PNG_PATH)

print("\nFinal constant-NAV plot complete.")


## Final output inventory

When run successfully, this notebook writes the following final artifacts:

| Output | Location |
|---|---|
| Selected trades CSV | `data/audit/locked_2621_win_band_25bps_conditional_selected_trades_v0_1.csv` |
| Selected trades parquet | `data/processed/locked_2621_win_band_25bps_conditional_selected_trades_v0_1.parquet` |
| Locked model summaries | `data/audit/locked_2621_win_band_25bps_conditional_*` |
| Constant $1mm NAV realized P&L curve | `data/audit/locked_2621_constant_1mm_nav_cumulative_pnl_realized_curve_v0_1.csv` |
| Constant $1mm NAV trade-level P&L | `data/audit/locked_2621_constant_1mm_nav_cumulative_pnl_trade_level_v0_1.csv` |
| Constant $1mm NAV cumulative P&L chart | `data/audit/locked_2621_constant_1mm_nav_cumulative_pnl_cumulative_pnl_v0_1.png` |

The constant-NAV chart is the final reporting view for this notebook: every trade is sized against $1mm regardless of prior gains or losses.
